In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv('data/model_data_revised.csv')

print(df.shape)
print(df["success"].value_counts(normalize=True))
print(df["release_year"].describe())



(6919, 41)
success
0    0.638098
1    0.361902
Name: proportion, dtype: float64
count    6919.000000
mean     2008.653418
std         9.298672
min      1990.000000
25%      2002.000000
50%      2009.000000
75%      2016.000000
max      2025.000000
Name: release_year, dtype: float64


## train-test split
#### cutoff year 2018

In [9]:
CUTOFF_YEAR = 2018

df_train = df[df["release_year"] < CUTOFF_YEAR].copy()
df_test = df[df["release_year"] >= CUTOFF_YEAR].copy()


## fixing missing runtime values & preprocessing
#### used median runtime
#### preprocessing: scaling numeric cols and onehot encoding categorial columns

In [18]:
df['runtime'].isna().sum()

np.int64(3)

In [20]:
target = 'success'

id_cols = ['tmdb_id', 'imdb_id', 'title_tmdb', 'release_date']

numeric_cols = ['runtime','release_year','n_genres','imdb_director_count','dir_prior_success_rate_smoothed','dir_prior_films_log1p',
                'dir_years_active_log1p','dir_prior_known_films_log1p','lead_actor_films_5yr_backfilled_log1p','company_films_5yr_log1p',
                'franchise_prior_films_log1p']

cat_cols = [
    'us_certification',
    'release_month',
]

# All remaining predictors are binary flags.
predictor_columns = [
    i for i in df.columns
    if i not in id_cols + [target]
]

binary_cols = [
    i for i in predictor_columns
    if i not in numeric_cols + cat_cols
]

# Separate predictors and target.
X_train = df_train[predictor_columns]
y_train = df_train[target]

X_test = df_test[predictor_columns]
y_test = df_test[target]

# impute missing runtime values using median  
# scales each numeric variable to have approx. mean 0 and sd 1  
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# one-hot encode categorial variables
categorical_pipeline = OneHotEncoder(
    handle_unknown="ignore"
)


preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_cols),
    ('categorical', categorical_pipeline, cat_cols),
    ('binary', 'passthrough', binary_cols),
])

# Fit preprocessing on training data only.
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same fitted preprocessing to the test data.
X_test_processed = preprocessor.transform(X_test)


## logit

In [24]:
# logistic regression
reg1 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000))
])

# Train the model
reg1.fit(X_train, y_train)

y_hat = reg1.predict(X_test)
y_prob = reg1.predict_proba(X_test)[:, 1]

# Calculate performance metrics
print('overall model metrics:')
print('accuracy', round(accuracy_score(y_test, y_hat), 3))
print('precision', round(precision_score(y_test, y_hat, zero_division=0), 3))
print('recall', round(recall_score(y_test, y_hat, zero_division=0), 3))
print('f1 score', round(f1_score(y_test, y_hat, zero_division=0), 3))
print('roc-auc score', round(roc_auc_score(y_test, y_prob), 3))

print('classification report (model metrics for specific groups)')
print(classification_report(y_test, y_hat, zero_division=0))

print('confusion matrix')
print(confusion_matrix(y_test, y_hat))

overall model metrics:
accuracy 0.66
precision 0.5
recall 0.505
f1 score 0.503
roc-auc score 0.685
classification report (model metrics for specific groups)
              precision    recall  f1-score   support

           0       0.74      0.74      0.74       900
           1       0.50      0.51      0.50       463

    accuracy                           0.66      1363
   macro avg       0.62      0.62      0.62      1363
weighted avg       0.66      0.66      0.66      1363

confusion matrix
[[666 234]
 [229 234]]


## random forest
### 300 trees, can also do one tree if we want but figured it wasn't necessary

In [26]:
forest1 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
forest1.fit(X_train, y_train)

# Make predictions
y_hat_forest = forest1.predict(X_test)
y_prob_forest = forest1.predict_proba(X_test)[:, 1]

In [ ]:
comparisons = {
    "Logistic Regression": (
        y_hat,
        y_prob
    ),
    "Random Forest": (y_hat_forest, y_prob_forest)
}

results = []

for name, (pred, prob) in comparisons.items():
    results.append({
        'Model Type': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1 Score': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC Score': roc_auc_score(y_test, prob)})

results_df = pd.DataFrame(results).set_index('Model Type')
results_df.round(3)

,Accuracy,Precision,Recall,F1 Score,ROC-AUC Score
Model Type,,,,,
Logistic Regression,0.660,0.500,0.505,0.503,0.685
Random Forest,0.676,0.536,0.354,0.427,0.676
